**Part 3: Policy RAG | Step 5: Faithfulness and abstention**  
Goal: check that answers are supported by the cited clauses, and that unanswerable questions get "not found".  

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

# Find the project root: the folder that contains src/policy_rag.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "policy_rag.py").exists())
sys.path.append(str(ROOT / "src"))
import policy_rag as rag

POLICY_DIR = ROOT / "notebooks" / "RAG" / "policy"
OUT_DIR = ROOT / "notebooks" / "RAG" / "outputs"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

**Step. Load the saved answers and the correct labels.**

In [2]:
results = json.loads((OUT_DIR / "answers.json").read_text(encoding="utf-8"))
eval_df = pd.read_csv(POLICY_DIR / "eval_questions.csv")
chunks = rag.chunk_policy(rag.load_policy(POLICY_DIR / "marketplace_policy.txt"))
index = rag.PolicyIndex(chunks)
len(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

20

**Step. Abstention and citation summary.** How the 4 unanswerable and 16 answerable questions were handled.

In [3]:
print(rag.summarize_answers(results).to_string())
print()
df = pd.DataFrame(results)
df[df["gold_answerable"] != df["answerable"].astype(int)][["qid", "gold_answerable", "answerable", "abstained_by", "answer"]]

unanswerable abstained (correct)       4 of 4
answerable wrongly abstained          1 of 16
answerable cited a correct clause    16 of 16



,qid,gold_answerable,answerable,abstained_by,answer
6,Q07,1,False,llm,"The provided clauses do not define the delay classes (Minor, Moder..."


**Inference**
- All 4 unanswerable questions (Q17 to Q20) were refused, so the "not found" rule worked on every near-miss question.
- 1 of 16 answerable questions was refused wrongly: Q07. The refusal came from the LLM (`abstained_by` = llm), not from the score threshold.
- The Q07 answer starts with "The provided clauses do not define the delay classes". The missing clause is `MOP-2.2`, the clause that defines the classes. This matches notebook 03, where the top 3 of hybrid search (`MOP-3.2`, `MOP-3.1`, `MOP-3.4`) did not contain it.
- The refusal is the correct behaviour for the context given: the model refused instead of guessing a delay class.
- "Cited a correct clause: 16 of 16" is a loose check. It counts Q07 too (refused, but it cited `MOP-3.2`), and it needs only one correct clause per question, not all of them.

**Step. The wrong answer in full.** The table above cuts long text after 70 characters. This prints the complete answer of every question where the result differs from the correct label.

In [4]:
wrong = [r for r in results if r["gold_answerable"] != int(r["answerable"])]
for r in wrong:
    print(r["qid"], "| abstained_by:", r["abstained_by"], "| retrieved:", ", ".join(r["retrieved_ids"]))
    print("  Q:", r["question"])
    print("  A:", r["answer"])
    print()

Q07 | abstained_by: llm | retrieved: MOP-3.2, MOP-3.1, MOP-3.4
  Q: An order worth R$ 500 is delivered 12 days late. How much platform credit does the customer get automatically?
  A: The provided clauses do not define the delay classes (Minor, Moderate, Severe, Critical), so a 12-day delay cannot be placed in a class. The credit therefore cannot be determined. MOP-3.2 gives the credit for each class. For a R$ 500 order, Minor is none, Moderate is the greater of R$ 20 or 10% (R$ 50), so R$ 50, and Severe is the greater of R$ 50 or 20% (R$ 100), so R$ 100. Critical is 100% of order value (R$ 500) plus freight, and the freight amount is not given. Under MOP-3.1 the compensation is issued as platform credit by default.



**Step. Faithfulness checks on answered questions.** Three checks: cited IDs are real, numbers in the answer come from the clauses or the question, and an LLM judge compares answer and clauses. This makes one LLM call per answered question.

In [5]:
checks = []
for r in results:
    c = rag.check_faithfulness(index, r)
    if c:
        checks.append({"qid": r["qid"], **c})
chk = pd.DataFrame(checks)
print(chk["verdict"].value_counts().to_string())
chk

verdict
SUPPORTED    14
PARTIAL       1


,qid,citations_valid,ungrounded_numbers,verdict,reason
0,Q01,True,[],SUPPORTED,Every step follows from the tier thresholds in 1.2 and the both-th...
1,Q02,True,[],SUPPORTED,The answer restates the 15-order threshold and the Unrated classif...
2,Q03,True,[],SUPPORTED,Clause 1.5 states that Gold sellers get a reduced commission rate ...
3,Q04,True,[],SUPPORTED,Every claim matches the clauses: 7 days is Moderate with a freight...
4,Q05,True,[2],SUPPORTED,"Grace allowance of 4 days applies to Northeast non-Express orders,..."
5,Q06,True,[],SUPPORTED,"The answer correctly applies 2.4 (no grace for Express), 2.2 (3 da..."
6,Q08,True,[],SUPPORTED,"The answer correctly applies both thresholds to R$ 1,500 (Head app..."
7,Q09,True,[],SUPPORTED,"Every claim in the answer matches clauses 3.1, 4.1 and 4.3: credit..."
8,Q10,True,[],SUPPORTED,Every claim matches the clauses: Pix refunds take 3 business days ...
9,Q11,True,[],SUPPORTED,Every claim in the answer matches clauses 5.2 and 2.3: removal is ...


**Inference**
- 15 answers were checked: the 15 questions that were answered. Q07 and the 4 unanswerable questions have no answer to check.
- 14 of 15 are SUPPORTED and 1 is PARTIAL (Q16). No answer is UNSUPPORTED.
- `citations_valid` is True for all 15, so every cited ID was a clause that was retrieved.
- The Q16 reason is cut off in the table. The full text is printed in Step 3b.

**Step. Faithfulness checks in full.** Prints every verdict and reason without cutting the text, and saves them to `outputs/faithfulness.json`. 

In [6]:
for _, row in chk.iterrows():
    print(f"{row['qid']} | {row['verdict']} | citations_valid={row['citations_valid']} | "
          f"ungrounded_numbers={row['ungrounded_numbers']}")
    print("  reason:", row["reason"])
    print()

chk.to_json(OUT_DIR / "faithfulness.json", orient="records", indent=2, force_ascii=False)

Q01 | SUPPORTED | citations_valid=True | ungrounded_numbers=[]
  reason: Every step follows from the tier thresholds in 1.2 and the both-thresholds rule and matching example in 1.3.

Q02 | SUPPORTED | citations_valid=True | ungrounded_numbers=[]
  reason: The answer restates the 15-order threshold and the Unrated classification for the trailing 90-day window, both of which are in the clause.

Q03 | SUPPORTED | citations_valid=True | ungrounded_numbers=[]
  reason: Clause 1.5 states that Gold sellers get a reduced commission rate of 11% against the standard 14%, which is exactly what the answer says.

Q04 | SUPPORTED | citations_valid=True | ungrounded_numbers=[]
  reason: Every claim matches the clauses: 7 days is Moderate with a freight chargeback, delay is measured against the checkout estimate, the waiver conditions and 10-day window apply, and no freight amount is stated. Omitting the 'or of the auto-cancellation date' option is not a contradiction, since auto-cancellation does not

**Step. Off-topic check.** Top cosine score of 3 questions that have nothing to do with the policy. If they score well below 0.68, the threshold can be set between them and the policy questions, so layer 1 blocks off-topic questions without touching any of the 20 test questions.

In [7]:
off_topic = ["What is the weather in Mumbai today?", "Who won the cricket world cup?", "How do I bake a cake?"]
for q in off_topic:
    print(round(float(index.dense_scores(q).max()), 3), "|", q)

0.426 | What is the weather in Mumbai today?
0.439 | Who won the cricket world cup?
0.424 | How do I bake a cake?


**Step. Final numbers for the report.**

In [8]:
per_q, s = rag.evaluate_retrieval(index, eval_df, ks=(1, 3, 5), mode="hybrid")
final = pd.concat([
    s.round(2).rename("value").to_frame(),
    rag.summarize_answers(results).rename("value").to_frame(),
    pd.Series({"faithfulness SUPPORTED": f"{(chk['verdict'] == 'SUPPORTED').sum()} of {len(chk)}",
               "answers with ungrounded numbers": f"{(chk['ungrounded_numbers'].str.len() > 0).sum()} of {len(chk)}"}).rename("value").to_frame(),
])
final

,value
any@1,0.81
all@1,0.5
any@3,1.0
all@3,0.94
any@5,1.0
all@5,1.0
MRR,0.91
unanswerable abstained (correct),4 of 4
answerable wrongly abstained,1 of 16
answerable cited a correct clause,16 of 16


**Inference**
- Retrieval (hybrid search, section title included): the top 3 holds every correct clause for 15 of 16 questions (`all@3` 0.94) and the top 5 for all 16 (`all@5` 1.00). A correct clause is ranked first for 13 of 16 (`any@1` 0.81), and `MRR` is 0.91.
- Generation: the answers in notebook 04 used the top 3 clauses of hybrid search. 15 of 16 answerable questions were answered. The one wrong refusal (Q07) is the one question that top 3 missed.
- Abstention: 4 of 4 unanswerable questions were refused.
- Faithfulness: 14 of 15 answers are SUPPORTED and 1 is PARTIAL. 1 of 15 has an ungrounded number (Q05, the computed value 2).
- Limits: the 20 questions were written from the policy with AI help, so scores may be optimistic. The judge and the answering model are the same model, which can be lenient on its own answers. The "cited a correct clause" check accepts one correct clause, not all.
- Next: re-run notebook 04 with dense search, section title and the top 5 clauses, then repeat this notebook to see if Q07 is fixed and the 4 unanswerable questions are still refused.

**Appendix. All 20 answers in full.** Every question with its retrieved clauses, citations, full answer and the expected answer.

In [9]:
exp = eval_df.set_index("qid")["expected_answer"]
for r in results:
    print(f"{r['qid']} | gold_answerable={r['gold_answerable']} | answered={r['answerable']} | "
          f"abstained_by={r['abstained_by']} | top_dense={r['top_dense']:.3f}")
    print("  retrieved:", ", ".join(r["retrieved_ids"]), "| cited:", ", ".join(r["citations"]) or "-")
    print("  Q:", r["question"])
    print("  A:", r["answer"])
    print("  Expected:", exp[r["qid"]])
    print()

Q01 | gold_answerable=1 | answered=True | abstained_by=None | top_dense=0.755
  retrieved: MOP-1.2, MOP-1.3, MOP-1.5 | cited: MOP-1.2, MOP-1.3
  Q: A seller has 96% on-time delivery and an 11% adverse review rate. Which tier is the seller in?
  A: The seller is in the Bronze tier. A seller must satisfy both thresholds for a tier (MOP-1.3). With 96% OTD, the seller meets the Gold OTD threshold (≥95%) but not the Gold ARR threshold (≤4%), since 11% exceeds 4%. Silver requires ARR ≤9%, and 11% exceeds that too. Bronze requires OTD ≥75% and ARR ≤15%, and the seller meets both (96% ≥ 75%, 11% ≤ 15%). This matches the example in MOP-1.3.
  Expected: Bronze. Both thresholds must be met, so 11% ARR rules out Gold.

Q02 | gold_answerable=1 | answered=True | abstained_by=None | top_dense=0.759
  retrieved: MOP-1.1, MOP-6.1, MOP-6.3 | cited: MOP-1.1
  Q: How many completed orders does a seller need in the 90-day window to be given a tier?
  A: A seller needs at least 15 completed orders in the tr